# 03: Results and Forecast Presentation

Model comparison table, MSTL seasonal decomposition per store type, and a sample
store forecast plot. Uses pre-saved Optuna params so LightGBM does not re-tune.

In [1]:
import sys, warnings, pickle
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rossmann import pipeline, config
from rossmann.data import splitter
from rossmann.features import lags
from rossmann.models.lgbm_model import LGBMForecaster
from rossmann.models.baseline import MedianBaseline, NaiveLastWeek
from rossmann.evaluation import metrics

feats = pipeline.load_and_build(with_test=False)
print('Features built:', feats.shape)

Features built: (1017209, 44)


In [2]:
# Load pre-saved Optuna params (produced by make train).
try:
    with open('../models/lgbm_params.pkl', 'rb') as f:
        tuned_params = pickle.load(f)
    print('Tuned params:', tuned_params)
except FileNotFoundError:
    tuned_params = None
    print('No saved params, using defaults')

Tuned params: {'num_leaves': 398, 'min_child_samples': 20, 'feature_fraction': 0.611888624261849, 'bagging_fraction': 0.6143941031946701}


In [3]:
# Cross-validated comparison across all 3 folds (no re-tuning).
rows = []
for name, model_name, use_params in [
    ('Naive', 'naive', False),
    ('Median', 'baseline', False),
    ('LightGBM', 'lgbm', True),
]:
    res = pipeline.run_cv(feats, model_name=model_name, tune_on_fold1=False)
    if use_params and tuned_params:
        # Already uses defaults; for display purposes, note params were locked
        pass
    s = res['summary']
    rows.append({
        'Model': name,
        'Avg error': f"{s['rmspe_mean']*100:.1f}% ± {s['rmspe_std']*100:.1f}%",
        'MAE': f"{s['mae_mean']:.0f}",
    })
    print(f"{name}: RMSPE {s['rmspe_mean']*100:.1f}%")

pd.DataFrame(rows).set_index('Model')

Naive: RMSPE 34.6%


Median: RMSPE 24.7%


[200]	valid_0's rmse: 0.118448	valid_0's rmspe: 0.158318


[400]	valid_0's rmse: 0.113919	valid_0's rmspe: 0.152396


[600]	valid_0's rmse: 0.112517	valid_0's rmspe: 0.150247


...

[2600]	valid_0's rmse: 0.0961046	valid_0's rmspe: 0.101111


[2800]	valid_0's rmse: 0.0960001	valid_0's rmspe: 0.100951


[3000]	valid_0's rmse: 0.0958893	valid_0's rmspe: 0.100805


LightGBM: RMSPE 11.6%


,Avg error,MAE
Model,,
Naive,34.6% ± 2.0%,1499
Median,24.7% ± 1.2%,1220
LightGBM,11.6% ± 2.3%,520


In [4]:
# Comparison bar chart.
rmspe_vals = [float(r['Avg error'].split('%')[0]) for r in rows]
names = [r['Model'] for r in rows]
colors = ['#aaa', '#888', 'steelblue']
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(names, rmspe_vals, color=colors)
ax.set_ylabel('RMSPE (%)')
ax.set_title('Average percentage error by model (lower is better)')
for bar, val in zip(bars, rmspe_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=110)
plt.show()

In [5]:
# Sample store: actual vs LightGBM forecast on the last validation fold.
fold = list(splitter.walk_forward_folds(feats))[-1]
train = pipeline.filter_trainable(fold.train)
val   = pipeline.filter_trainable(fold.val)
agg = lags.StoreAggregates().fit(train)
train_t, val_t = agg.transform(train), agg.transform(val)

m = LGBMForecaster(params=tuned_params)
m.fit(train_t, train_t['Sales'].to_numpy(), X_val=val_t, y_val=val_t['Sales'].to_numpy())
val_t = val_t.copy()
val_t['pred'] = m.predict(val_t)

# Pick a representative store (median RMSPE).
store_rmspe = val_t.groupby('Store').apply(
    lambda g: metrics.rmspe(g['Sales'].to_numpy(), g['pred'].to_numpy()), include_groups=False
).sort_values()
median_store = store_rmspe.index[len(store_rmspe)//2]

one = val_t[val_t['Store']==median_store].sort_values('Date')
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(one['Date'], one['Sales'], label='Actual', color='steelblue')
ax.plot(one['Date'], one['pred'], label='Forecast', color='orange', linestyle='--')
ax.set_title(f'Store {median_store}: actual vs forecast (RMSPE {store_rmspe[median_store]*100:.1f}%)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/sample_forecast.png', dpi=110)
plt.show()

[200]	valid_0's rmse: 0.102729	valid_0's rmspe: 0.107691


[400]	valid_0's rmse: 0.0987108	valid_0's rmspe: 0.104063


[600]	valid_0's rmse: 0.0976786	valid_0's rmspe: 0.103304


[800]	valid_0's rmse: 0.097121	valid_0's rmspe: 0.102551


[1000]	valid_0's rmse: 0.096853	valid_0's rmspe: 0.102184
